# 🏋️ Stamp Detection Model Training - Google Colab

This notebook trains a YOLOv8 segmentation model for stamp detection.

## Instructions
1. Set runtime to GPU (Runtime → Change runtime type → T4 GPU)
2. Upload your dataset to Google Drive
3. Run all cells in order

## 1. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install ultralytics

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 2. Configure Paths

Update the path below to match your Google Drive structure.

In [ ]:
import os

# UPDATE THIS PATH to your project location in Google Drive
PROJECT_PATH = '/content/drive/MyDrive/stamp-detection-claude'

# Navigate to project
%cd {PROJECT_PATH}

# Verify dataset exists
dataset_yaml = os.path.join(PROJECT_PATH, 'dataset/data.yaml')
if os.path.exists(dataset_yaml):
    print(f"✅ Dataset config found: {dataset_yaml}")
else:
    print(f"❌ Dataset config not found at: {dataset_yaml}")
    print("Please ensure your dataset is uploaded correctly.")

## 3. Training Configuration

Adjust these parameters as needed.

In [ ]:
# Training parameters
CONFIG = {
    'base_model': 'yolov8n-seg.pt',  # Nano model (fastest)
    # 'base_model': 'yolov8s-seg.pt',  # Small model (better accuracy)
    # 'base_model': 'yolov8m-seg.pt',  # Medium model (best balance)
    
    'epochs': 100,           # Number of training epochs
    'batch_size': 16,       # Batch size (reduce if OOM)
    'img_size': 640,        # Image size
    'patience': 50,         # Early stopping patience
    
    # Augmentation (built into YOLOv8)
    'degrees': 15,          # Rotation augmentation
    'mosaic': 1.0,          # Mosaic augmentation probability
    'mixup': 0.1,           # Mixup augmentation probability
}

print("Training configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 4. Start Training

This will take 1-4 hours depending on dataset size and GPU.

In [ ]:
from ultralytics import YOLO

# Load base model
model = YOLO(CONFIG['base_model'])

# Start training
results = model.train(
    data='dataset/data.yaml',
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['img_size'],
    batch=CONFIG['batch_size'],
    patience=CONFIG['patience'],
    device=0,  # Use GPU
    
    # Augmentation
    degrees=CONFIG['degrees'],
    mosaic=CONFIG['mosaic'],
    mixup=CONFIG['mixup'],
    
    # Output
    project='runs/segment',
    name='stamp_train',
    exist_ok=True,
)

print("\n✅ Training complete!")

## 5. Evaluate Model

In [ ]:
# Validate model
metrics = model.val()

print("\n📊 Validation Results:")
print(f"  mAP50: {metrics.seg.map50:.4f}")
print(f"  mAP50-95: {metrics.seg.map:.4f}")

## 6. Save Model

Copy the trained model to your project's models folder.

In [ ]:
import shutil

# Source and destination paths
source_path = 'runs/segment/stamp_train/weights/best.pt'
dest_folder = 'models/stamp_detector_seg/weights'
dest_path = os.path.join(dest_folder, 'best.pt')

# Create destination folder
os.makedirs(dest_folder, exist_ok=True)

# Copy model
shutil.copy(source_path, dest_path)

print(f"✅ Model saved to: {dest_path}")
print(f"\nModel size: {os.path.getsize(dest_path) / 1024 / 1024:.2f} MB")

## 7. Test Inference

Test the model on a sample image.

In [ ]:
# Run inference on test images
test_images = 'dataset/test/images'  # Update if needed

if os.path.exists(test_images):
    results = model.predict(
        source=test_images,
        save=True,
        conf=0.5,
        project='runs/predict',
        name='test_inference'
    )
    print(f"✅ Predictions saved to: runs/predict/test_inference")
else:
    print(f"Test images folder not found: {test_images}")

## 8. Download Model (Optional)

Download the trained model to your local machine.

In [ ]:
from google.colab import files

# Download best.pt
files.download('models/stamp_detector_seg/weights/best.pt')

## Training Complete! 🎉

Your trained model is now saved in:
- Google Drive: `models/stamp_detector_seg/weights/best.pt`
- Also available in Colab: `runs/segment/stamp_train/weights/best.pt`

### Next Steps:
1. Download the model to your local machine
2. Place it in your project's `models/stamp_detector_seg/weights/` folder
3. Run the GUI to test your new model

### Tips:
- If mAP is low, try more epochs or a larger base model
- If training is slow, reduce batch size
- For better accuracy, use more training data